In [ ]:
pip install langchain langchain-openai

In [2]:

# IMPORTS

import os
import wikipedia

from langchain_openai import ChatOpenAI
from langchain.agents.factory import create_agent
from langchain_core.messages import HumanMessage

In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7,
    streaming=True
)

# This environment uses the newer LangChain agent API, so conversation memory is handled by the graph state.


In [4]:
def paquetes_turisticos(zona: str) -> str:
    """
    Entrega paquetes turísticos de Puerto Montt y alrededores.
    """

    zona = zona.lower()

    paquetes = {

        "puerto montt": [
            "Tour Angelmó + Costanera",
            "Tour Alerce Andino",
            "Tour Reloncaví"
        ],

        "puerto varas": [
            "Tour Lago Llanquihue",
            "Tour Saltos del Petrohué",
            "Tour Volcán Osorno"
        ],

        "chiloe": [
            "Tour Castro",
            "Tour Iglesias Patrimoniales",
            "Tour Pingüineras"
        ],

        "frutillar": [
            "Tour Teatro del Lago",
            "Tour Costanera de Frutillar",
            "Tour Cultural Alemán"
        ]
    }

    if zona in paquetes:

        respuesta = f"Paquetes disponibles en {zona}:\n"

        for paquete in paquetes[zona]:
            respuesta += f"- {paquete}\n"

        return respuesta

    return "No hay paquetes registrados para esa zona."

def precio_tour(personas: int) -> str:
    """
    Calcula el precio total de un tour.
    """

    valor_persona = 50000
    total = personas * valor_persona

    return f"El precio total para {personas} personas es ${total} pesos chilenos."

def informacion_3m_tours(texto: str) -> str:
    """
    Entrega información sobre la empresa.
    """

    return """
    3M Tours es una agencia de turismo ubicada en Puerto Montt.
    Ofrece tours y paquetes turísticos en la Región de Los Lagos.
    Especialistas en recorridos naturales, volcanes, lagos y turismo aventura.
    """

tools = [paquetes_turisticos, precio_tour, informacion_3m_tours]

In [5]:
# AGENTE

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Eres un asistente de 3M Tours. Responde preguntas sobre tours, paquetes y precios en la Región de Los Lagos.",
    debug=False
)



In [6]:
#Prueba

print("------------------------------------")
print("      BIENVENIDO A 3M TOURS")
print("------------------------------------")

while True:

    pregunta = input("\nCliente: ")

    if pregunta.lower() == "salir":
        print("Gracias por preferir 3M Tours.")
        break

    respuesta = agent.invoke(
        {
            "messages": [
                HumanMessage(content=pregunta)
            ]
        }
    )

    messages = respuesta.messages if hasattr(respuesta, "messages") else respuesta["messages"]
    agente_text = messages[-1].content if messages else str(respuesta)

    print("\nAgente:")
    print(agente_text)


------------------------------------
      BIENVENIDO A 3M TOURS
------------------------------------

Agente:
¡Hola! ¿En qué puedo ayudarte hoy? Si tienes preguntas sobre tours, paquetes o precios en la Región de Los Lagos, no dudes en preguntar.

Agente:
En Puerto Montt puedes visitar los siguientes lugares a través de nuestros paquetes turísticos:

1. **Tour Angelmó + Costanera**: Disfruta de la belleza del puerto y sus alrededores, incluyendo el famoso mercado de Angelmó.
   
2. **Tour Alerce Andino**: Explora el Parque Nacional Alerce Andino, conocido por sus impresionantes paisajes y la flora nativa.

3. **Tour Reloncaví**: Conoce la belleza de la Bahía de Reloncaví y sus alrededores, ideal para los amantes de la naturaleza.

Si deseas más información sobre alguno de estos tours, no dudes en preguntar.

Agente:
Actualmente no tenemos paquetes turísticos disponibles específicamente para el Volcán Osorno. Si estás interesado en otros destinos o actividades en la Región de Los Lagos